## Prerequisites

Runtime: Python 3, T4 GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%pip install -q "git+https://github.com/huggingface/transformers.git" accelerate bitsandbytes Pillow

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

In [4]:
import torch

In [5]:
from pathlib import Path

WORKING_DIR = Path('/content/drive/MyDrive/aiOCR')
MODEL_NAME = 'google/gemma-4-E4B-it'

qc = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=qc,
    device_map="auto",
    torch_dtype=torch.float16,
).eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

In [6]:
IMAGE_FILE = WORKING_DIR / 'images/pineda1/pineda1_page_3.png'

## Inference

In [ ]:
import time
from PIL import Image

image_stem   = IMAGE_FILE.stem
image_folder = IMAGE_FILE.parent.name

ENABLE_THINKING = True
MAX_SIDE = 1024  # cap before handing to the processor to avoid tiling artefacts on T4

prompt = (
    "Convert the document to plain text, as close to the original as possible "
    "(including typos, print errors, and original grammar and spelling). "
    "Do not add any formatting, markdown, or annotations."
)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt},
        ],
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)

image = Image.open(IMAGE_FILE).convert("RGB")
# Downscale so the longest side = MAX_SIDE, preserving aspect ratio
if max(image.size) > MAX_SIDE:
    image.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)

inputs = processor(
    text=text,
    images=[image],
    return_tensors="pt",
    max_num_tokens=560,  # 560 keeps detail high while fitting T4 VRAM safely
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

t0 = time.time()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=4096,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        do_sample=True,
    )
elapsed = time.time() - t0

raw_response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
transcription = processor.parse_response(raw_response)

print(f"Done in {elapsed:.1f}s")
print(transcription)


Done in 75.4s
{'role': 'assistant', 'thinking': 'Thinking Process:\n\n1.  **Analyze the Request:** The user wants me to convert the provided image/document content into plain text.\n2.  **Constraint Checklist:**\n    *   Convert to plain text. (Yes)\n    *   As close to the original as possible. (Yes)\n    *   Include typos, print errors, and original grammar/spelling. (Yes)\n    *   Do not add any formatting, markdown, or annotations. (Crucial: Strict adherence to plain text output).\n3.  **Analyze the Input Image:** The image provided is largely a solid gray field. There is extremely small, faint, or illegible text at the top and bottom, which appears to be metadata or a signature/watermark (e.g., "stylian hyplan hyplan hyplan").\n4.  **Transcription Strategy:** Transcribe the visible text elements while maintaining the structure and content as they appear. Since the majority of the image is blank gray, that should be reflected if possible, but usually, "converting to plain text" foc

### Saving the output

In [13]:
# Transcription → transcriptions/gemma-4-E4B/<stem>.md
transcription_out = WORKING_DIR / 'transcriptions/gemma-4-E4B'
transcription_out.mkdir(parents=True, exist_ok=True)

(transcription_out / f'{image_stem}.md').write_text(transcription['content'], encoding='utf-8')
print(f"Saved: transcriptions/gemma-4-E4B/{image_stem}.md")

if ENABLE_THINKING:
    # Reasoning → reasonings/gemma-4-E4B/<stem>.md
    reasoning_out = WORKING_DIR / 'reasonings/gemma-4-E4B'
    reasoning_out.mkdir(parents=True, exist_ok=True)
    frontmatter = (
        "---\n"
        f"model: {MODEL_NAME}\n"
        f"prompt: {prompt}\n"
        f"enable_thinking: {ENABLE_THINKING}\n"
        f"attachement: images/{image_folder}/{IMAGE_FILE.name}\n"
        "---\n\n"
    )
    (reasoning_out / f'{image_stem}.md').write_text(frontmatter + raw_response, encoding='utf-8')
    print(f"Saved: reasonings/gemma-4-E4B/{image_stem}.md")

Saved: transcriptions/gemma-4-E4B/pineda1_page_3.md
Saved: reasonings/gemma-4-E4B/pineda1_page_3.md
